# Stage 5: $k_0$ pH-rate analysis for water/hydroxide background kinetics

This notebook aggregates the `k0` intercepts from the first-stage `kobs` vs
`[Nu]free` linear regressions generated by `stage2_4_mechanistic_analysis.ipynb`.

The interpretation is that these intercepts describe the background reaction
of the phosphorylating agent in the absence of the added nucleophile. Aggregated
over many nucleophiles and pH values, they provide a pH-rate profile for the
water/hydroxide contribution.

Only the fitted `k0` value from the linear regression is used as the dependent
variable in the pH-rate fit. Bootstrap confidence intervals of the first-stage
`k0` values are used for input quality filtering and as point error bars in the
plots, but not as fit weights.

## Analysis Strategy

For every `*_k2_next_step_analysis.xlsx` workbook in the `P5`-`P9`/`S*` subdirectories:

1. Read the `strategy_A_kobs_to_k2_app` sheet.
2. Extract `electrophile_type`, `pH_corr_ionize_mean`, and the linear-regression
   intercept `k0`.
3. Keep finite positive `k0` values with sufficient inlier support and acceptable
   bootstrap confidence intervals.
4. Aggregate all retained `k0` values by phosphorylating agent.
5. Fit the fixed-pKa two-state pH-rate model:

```text
k0(pH) = (1 - alpha) * k0_low + alpha * k0_high
alpha = 1 / (1 + 10^(pKa - pH))
```

The fit uses the same two-step robust idea as the nucleophile pH-rate analysis:
an initial robust fit defines residuals, a MAD cutoff identifies heavy outliers,
and the final fit is performed on inliers. Confidence intervals are estimated
by Bayesian bootstrapping.

In [1]:
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=RuntimeWarning)

NOTEBOOK_DIR = Path.cwd().resolve()
KINETICS_DIR = Path(os.environ.get("KINETICS_DIR", NOTEBOOK_DIR)).expanduser().resolve()

P_KA_WORKBOOK = Path(os.environ.get(
    "PHOSPHORYLATING_PKA_PATH",
    KINETICS_DIR / "phosphorylating_pka.xlsx",
)).expanduser()

BOOTSTRAP_N = int(os.environ.get("KINETICS_BOOTSTRAP_N", "2000"))
RANDOM_SEED = int(os.environ.get("KINETICS_RANDOM_SEED", "20260708"))
MIN_POINTS = 3  # Minimum number of observations required for fitting
OUTLIER_FACTOR = 3.0  # Robust-residual multiplier used to identify outliers
K0_MIN_INLIERS = 3  # Minimum number of inlier observations required for accepting a k0 estimate
REQUIRE_STRATEGY_A_SUCCESS = False  # Whether k0 inputs must come from successful Strategy A k2,app fits
K0_REQUIRE_LO95_POSITIVE = True  # Whether the lower bound of the k0 95% CI must be positive
K0_MAX_REL_CI_WIDTH = 3.0  # Maximum relative width of the k0 95% CI for accepting a k0 estimate

rng = np.random.default_rng(RANDOM_SEED)

print(f"KINETICS_DIR = {KINETICS_DIR}")
print(f"P_KA_WORKBOOK = {P_KA_WORKBOOK}")
print(f"BOOTSTRAP_N = {BOOTSTRAP_N}")
print(f"REQUIRE_STRATEGY_A_SUCCESS = {REQUIRE_STRATEGY_A_SUCCESS}")
print(f"K0_REQUIRE_LO95_POSITIVE = {K0_REQUIRE_LO95_POSITIVE}")
print(f"K0_MAX_REL_CI_WIDTH = {K0_MAX_REL_CI_WIDTH}")


KINETICS_DIR = /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics
P_KA_WORKBOOK = /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/phosphorylating_pka.xlsx
BOOTSTRAP_N = 2000
REQUIRE_STRATEGY_A_SUCCESS = False
K0_REQUIRE_LO95_POSITIVE = True
K0_MAX_REL_CI_WIDTH = 3.0


## Helper Functions

In [2]:
def sanitize_filename(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unknown"


def electrophile_key(value):
    match = re.match(r"(P\d+)", str(value).strip(), flags=re.IGNORECASE)
    return match.group(1).upper() if match else str(value).strip()


def discover_px_directories(root=KINETICS_DIR):
    out = {}
    for path in sorted(root.glob("P*")):
        if path.is_dir():
            out.setdefault(electrophile_key(path.name), path)
    return out


PX_OUTPUT_DIRS = discover_px_directories()


def resolve_px_output_dir(electrophile):
    key = electrophile_key(electrophile)
    if key in PX_OUTPUT_DIRS:
        return PX_OUTPUT_DIRS[key]
    fallback = KINETICS_DIR / sanitize_filename(key)
    fallback.mkdir(parents=True, exist_ok=True)
    PX_OUTPUT_DIRS[key] = fallback
    return fallback


def format_k0_workbook(writer):
    wb = writer.book
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for column_cells in ws.columns:
            header = column_cells[0].value
            width = min(max(len(str(header)) + 2, 10), 42)
            ws.column_dimensions[column_cells[0].column_letter].width = width
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, float):
                    if "k0" in str(ws.cell(1, cell.column).value).lower():
                        cell.number_format = "0.00E+00"
                    else:
                        cell.number_format = "0.000"


def write_k0_excel_workbook(path, summary_df, values_df, raw_df, pka_df, metadata_df, read_errors_df=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        values_df.to_excel(writer, sheet_name="k0_values_used", index=False)
        raw_df.to_excel(writer, sheet_name="all_k0_values", index=False)
        pka_df.to_excel(writer, sheet_name="electrophile_pKa", index=False)
        metadata_df.to_excel(writer, sheet_name="metadata", index=False)
        if read_errors_df is not None and len(read_errors_df):
            read_errors_df.to_excel(writer, sheet_name="read_errors", index=False)
        format_k0_workbook(writer)


def alpha_deprotonated(pH, pKa):
    pH = np.asarray(pH, dtype=float)
    return 1.0 / (1.0 + 10.0 ** (pKa - pH))


def k0_ph_rate_model(log_params, pH, pKa):
    # Fit in log-parameter space to enforce positive k0_low and k0_high.
    k0_low, k0_high = np.exp(log_params)
    a = alpha_deprotonated(pH, pKa)
    return (1.0 - a) * k0_low + a * k0_high


def log_residuals(log_params, pH, y, pKa, weights=None):
    pred = k0_ph_rate_model(log_params, pH, pKa)
    pred = np.clip(pred, 1e-300, np.inf)
    y = np.clip(np.asarray(y, dtype=float), 1e-300, np.inf)
    resid = np.log10(pred) - np.log10(y)
    if weights is not None:
        resid = resid * np.sqrt(weights)
    return resid


def r2_log_score(y_obs, y_pred):
    y_obs = np.log10(np.clip(np.asarray(y_obs, dtype=float), 1e-300, np.inf))
    y_pred = np.log10(np.clip(np.asarray(y_pred, dtype=float), 1e-300, np.inf))
    ss_res = np.sum((y_obs - y_pred) ** 2)
    ss_tot = np.sum((y_obs - np.mean(y_obs)) ** 2)
    return np.nan if ss_tot == 0 else 1.0 - ss_res / ss_tot


def robust_mad_scale(resid):
    resid = np.asarray(resid, dtype=float)
    med = np.nanmedian(resid)
    mad = np.nanmedian(np.abs(resid - med))
    if not np.isfinite(mad) or mad == 0:
        scale = np.nanstd(resid)
    else:
        scale = 1.4826 * mad
    if not np.isfinite(scale) or scale == 0:
        scale = 1e-12
    return med, scale


def initial_log_params(y):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y) & (y > 0)]
    if len(y) == 0:
        return np.log([1e-8, 1e-8])
    lo = max(np.nanpercentile(y, 25), 1e-30)
    hi = max(np.nanpercentile(y, 75), 1e-30)
    return np.log([lo, hi])


def fit_k0_ph_rate_two_step(pH, y, pKa, y_lo=None, y_hi=None):
    pH = np.asarray(pH, dtype=float)
    y = np.asarray(y, dtype=float)
    y_lo = np.full_like(y, np.nan, dtype=float) if y_lo is None else np.asarray(y_lo, dtype=float)
    y_hi = np.full_like(y, np.nan, dtype=float) if y_hi is None else np.asarray(y_hi, dtype=float)
    mask = np.isfinite(pH) & np.isfinite(y) & (y > 0)
    pH = pH[mask]
    y = y[mask]
    y_lo = y_lo[mask]
    y_hi = y_hi[mask]

    if len(y) < MIN_POINTS:
        return {"success": False, "failure_reason": "fewer than 3 positive k0 values"}

    x0 = initial_log_params(y)
    first = least_squares(
        log_residuals, x0, args=(pH, y, pKa),
        loss="huber", f_scale=0.20, max_nfev=20000,
    )
    resid0 = log_residuals(first.x, pH, y, pKa)
    resid_median, resid_scale = robust_mad_scale(resid0)
    inlier_mask = np.abs(resid0 - resid_median) <= OUTLIER_FACTOR * resid_scale

    if inlier_mask.sum() < MIN_POINTS:
        return {
            "success": False,
            "failure_reason": "fewer than 3 inliers after Huber/MAD filtering",
            "params_initial": first.x,
            "inlier_mask": inlier_mask,
            "resid_initial": resid0,
            "pH": pH,
            "y": y,
            "y_lo": y_lo,
            "y_hi": y_hi,
        }

    second = least_squares(
        log_residuals, first.x,
        args=(pH[inlier_mask], y[inlier_mask], pKa),
        loss="linear", max_nfev=20000,
    )
    pred_all = k0_ph_rate_model(second.x, pH, pKa)
    pred_in = k0_ph_rate_model(second.x, pH[inlier_mask], pKa)
    resid_in = log_residuals(second.x, pH[inlier_mask], y[inlier_mask], pKa)

    return {
        "success": True,
        "params": second.x,
        "k0_low": float(np.exp(second.x[0])),
        "k0_high": float(np.exp(second.x[1])),
        "inlier_mask": inlier_mask,
        "resid_initial": resid0,
        "resid_median_initial": resid_median,
        "resid_scale_initial": resid_scale,
        "pH": pH,
        "y": y,
        "y_lo": y_lo,
        "y_hi": y_hi,
        "pred_all": pred_all,
        "r2_log": r2_log_score(y[inlier_mask], pred_in),
        "rmse_log": float(np.sqrt(np.mean(resid_in ** 2))),
        "n_total": int(len(y)),
        "n_inliers": int(inlier_mask.sum()),
        "n_outliers": int((~inlier_mask).sum()),
    }


def bayesian_bootstrap_k0_ph_rate(fit, pKa, n_boot=BOOTSTRAP_N):
    if not fit.get("success", False):
        return pd.DataFrame()

    pH = fit["pH"][fit["inlier_mask"]]
    y = fit["y"][fit["inlier_mask"]]
    x0 = fit["params"]
    rows = []
    skipped = 0

    for _ in range(n_boot):
        weights = rng.dirichlet(np.ones(len(y))) * len(y)
        try:
            res = least_squares(
                log_residuals, x0,
                args=(pH, y, pKa, weights),
                loss="linear", max_nfev=10000,
            )
            k0_low, k0_high = np.exp(res.x)
            if np.isfinite(k0_low) and np.isfinite(k0_high):
                rows.append({"k0_low": k0_low, "k0_high": k0_high})
            else:
                skipped += 1
        except Exception:
            skipped += 1

    out = pd.DataFrame(rows)
    out.attrs["skipped"] = skipped
    return out


def quantile_ci(series):
    if len(series) == 0:
        return (np.nan, np.nan)
    return tuple(np.nanpercentile(series, [2.5, 97.5]))


def choose_y_scale(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if len(values) == 0:
        return 1.0, "s^-1"
    target = np.nanmedian(values)
    if target <= 0:
        return 1.0, "s^-1"
    exponent = int(np.floor(np.log10(target)))
    scale = 10.0 ** exponent
    return scale, rf"$10^{{{exponent}}}$ s$^{{-1}}$"


def plot_k0_profile(electrophile, pKa, fit, boots, high_summary, path):
    pH = fit["pH"]
    y = fit["y"]
    y_lo = fit.get("y_lo", np.full_like(y, np.nan, dtype=float))
    y_hi = fit.get("y_hi", np.full_like(y, np.nan, dtype=float))
    inlier = fit["inlier_mask"]

    x_min = np.nanmin(pH) - 0.25
    x_max = np.nanmax(pH) + 0.25
    xg = np.linspace(x_min, x_max, 300)
    yg = k0_ph_rate_model(fit["params"], xg, pKa)

    ci_lo = ci_hi = None
    if boots is not None and len(boots):
        curves = []
        for _, row in boots.iterrows():
            curves.append((1 - alpha_deprotonated(xg, pKa)) * row["k0_low"] + alpha_deprotonated(xg, pKa) * row["k0_high"])
        curves = np.asarray(curves)
        ci_lo = np.nanpercentile(curves, 2.5, axis=0)
        ci_hi = np.nanpercentile(curves, 97.5, axis=0)

    all_y = np.concatenate([y, yg, ci_lo if ci_lo is not None else yg, ci_hi if ci_hi is not None else yg])
    scale, ylabel_scale = choose_y_scale(all_y)

    plt.rcParams.update({
        "font.size": 12,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "legend.fontsize": 12,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
    })
    fig, ax = plt.subplots(figsize=(13.0, 5.6), dpi=180)

    if ci_lo is not None:
        ax.fill_between(xg, ci_lo / scale, ci_hi / scale, color="#bfe7ea", alpha=0.75, linewidth=0)
    ax.plot(xg, yg / scale, color="#22344a", linewidth=2.6)

    yerr_lo = np.where(np.isfinite(y_lo), np.maximum(y - y_lo, 0), np.nan) / scale
    yerr_hi = np.where(np.isfinite(y_hi), np.maximum(y_hi - y, 0), np.nan) / scale
    yerr = np.vstack([yerr_lo, yerr_hi])

    ax.errorbar(
        pH[inlier], y[inlier] / scale, yerr=yerr[:, inlier],
        fmt="o", markersize=6.5, markerfacecolor="white",
        markeredgecolor="#477fa3", markeredgewidth=1.8,
        ecolor="#477fa3", elinewidth=1.0, capsize=2.5,
        linestyle="none", zorder=3,
    )
    if np.any(~inlier):
        ax.errorbar(
            pH[~inlier], y[~inlier] / scale, yerr=yerr[:, ~inlier],
            fmt="D", markersize=6.5, markerfacecolor="white",
            markeredgecolor="#ef3b4b", markeredgewidth=1.8,
            ecolor="#ef3b4b", elinewidth=1.0, capsize=2.5,
            linestyle="none", zorder=4,
        )

    if np.isfinite(pKa):
        ax.axvline(pKa, color="0.70", linestyle=":", linewidth=1.6)

    if high_summary and np.isfinite(high_summary.get("median", np.nan)):
        median = high_summary["median"]
        sd = high_summary.get("sd", np.nan)
        ax.axhline(median / scale, color="#ef3b4b", linestyle="--", linewidth=2.0)
        if np.isfinite(sd):
            ax.axhspan((median - sd) / scale, (median + sd) / scale, color="#ef3b4b", alpha=0.08)

    ax.set_title(f"Background k0 pH-rate profile: {electrophile}")
    ax.set_xlabel("mean pH$_{corr,ionize}$")
    ax.set_ylabel(rf"$k_0$ / {ylabel_scale}")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    text = (
        f"k0,low = {fit['k0_low']:.2e}\n"
        f"k0,high = {fit['k0_high']:.2e}\n"
        f"R$^2_{{log}}$ = {fit['r2_log']:.3f}\n"
        f"RMSE$_{{log}}$ = {fit['rmse_log']:.3f}\n"
        f"n = {fit['n_total']} ({fit['n_inliers']} in)\n"
        f"pKa = {pKa:.2f}"
    )
    if boots is not None and len(boots):
        low_lo, low_hi = quantile_ci(boots["k0_low"])
        high_lo, high_hi = quantile_ci(boots["k0_high"])
        text += f"\n95% CI k0,low [{low_lo:.1e}, {low_hi:.1e}]"
        text += f"\n95% CI k0,high [{high_lo:.1e}, {high_hi:.1e}]"

    ax.text(
        1.01, 1.00, text, transform=ax.transAxes,
        va="top", ha="left", color="#22344a", fontsize=11,
        bbox=dict(facecolor="white", edgecolor="#b7d4df", linewidth=1.2, boxstyle="round,pad=0.35"),
    )

    handles = [
        Patch(facecolor="#bfe7ea", edgecolor="none", alpha=0.75, label="95% bootstrap CI"),
        Line2D([0], [0], color="#22344a", linewidth=2.6, label="pH-rate fit"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="white", markeredgecolor="#477fa3", markeredgewidth=1.8, markersize=7, label="inliers"),
        Line2D([0], [0], marker="D", color="none", markerfacecolor="white", markeredgecolor="#ef3b4b", markeredgewidth=1.8, markersize=7, label="outliers"),
        Line2D([0], [0], color="#ef3b4b", linestyle="--", linewidth=2.0, label="median high-pH k0"),
    ]
    ax.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 0.44),
        borderaxespad=0.0,
        frameon=True,
    )
    fig.tight_layout(rect=(0, 0, 0.74, 1))
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


## Read k0 Values From Stage 2-4 Analysis Workbooks

In [3]:
def read_pka_table(path):
    df = pd.read_excel(path)
    cols = {c.lower(): c for c in df.columns}
    px_col = cols.get("px") or cols.get("electrophile_type")
    pka_col = cols.get("pka")
    if px_col is None or pka_col is None:
        raise ValueError(f"Could not identify PX/pKa columns in {path}")
    out = df[[px_col, pka_col]].rename(columns={px_col: "electrophile_type", pka_col: "electrophile_pKa"})
    out["electrophile_type"] = out["electrophile_type"].astype(str).str.strip()
    out["electrophile_pKa"] = pd.to_numeric(out["electrophile_pKa"], errors="coerce")
    return out.dropna(subset=["electrophile_type", "electrophile_pKa"])


def read_k0_workbook(path):
    df = pd.read_excel(path, sheet_name="strategy_A_kobs_to_k2_app")
    # path layout: <root>/<PX>/<S*_NSxxx>/k2_next_step_analysis/<file>.xlsx
    px_dir = path.parent.parent.parent.name
    experiment_dir = path.parent.parent.name

    required = ["electrophile_type", "ha_initial_pH", "pH_corr_ionize_mean", "k0", "k0_lo95", "k0_hi95", "n_selected", "n_inliers"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    keep_cols = [
        "electrophile_type", "nu_id", "nu_name", "ha_initial_pH", "pH_corr_ionize_mean",
        "pH_corr_ionize_abs_drift_max", "k_obs_values_used", "n_selected", "n_inliers",
        "n_outliers", "outlier_entries", "k2_app", "k0", "k0_lo95", "k0_hi95", "r2", "rmse",
        "analysis_success", "failure_reason",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    out = df[keep_cols].copy()
    out["PX_dir"] = px_dir
    out["experiment"] = experiment_dir
    out["source_workbook"] = str(path)
    out["k0_linear"] = pd.to_numeric(out["k0"], errors="coerce")
    out["k0_lo95"] = pd.to_numeric(out["k0_lo95"], errors="coerce")
    out["k0_hi95"] = pd.to_numeric(out["k0_hi95"], errors="coerce")
    out["k0_rel_ci_width"] = (out["k0_hi95"] - out["k0_lo95"]) / out["k0_linear"].abs()
    out["pH"] = pd.to_numeric(out["pH_corr_ionize_mean"], errors="coerce")
    out["n_inliers"] = pd.to_numeric(out["n_inliers"], errors="coerce")
    out["n_selected"] = pd.to_numeric(out["n_selected"], errors="coerce")
    out["analysis_success"] = out["analysis_success"].fillna(False).astype(bool)
    return out


pka_df = read_pka_table(P_KA_WORKBOOK)
workbooks = sorted(KINETICS_DIR.glob("P*/S*/k2_next_step_analysis/*_k2_next_step_analysis.xlsx"))
print(f"Found {len(workbooks)} Stage 2-4 result workbooks")

frames = []
errors = []
for path in workbooks:
    try:
        frames.append(read_k0_workbook(path))
    except Exception as exc:
        errors.append({"source_workbook": str(path), "error": str(exc)})

if not frames:
    raise RuntimeError("No k0 values could be read from the Stage 2-4 result workbooks.")

raw_k0 = pd.concat(frames, ignore_index=True)
raw_k0 = raw_k0.merge(pka_df, on="electrophile_type", how="left")
raw_k0["k0_ci_finite"] = np.isfinite(raw_k0["k0_lo95"]) & np.isfinite(raw_k0["k0_hi95"])
raw_k0["k0_ci_ordered"] = raw_k0["k0_hi95"] >= raw_k0["k0_lo95"]
raw_k0["k0_ci_positive"] = raw_k0["k0_lo95"] > 0
raw_k0["k0_input_usable"] = (
    np.isfinite(raw_k0["k0_linear"])
    & (raw_k0["k0_linear"] > 0)
    & raw_k0["k0_ci_finite"]
    & raw_k0["k0_ci_ordered"]
    & ((raw_k0["k0_ci_positive"]) | (not K0_REQUIRE_LO95_POSITIVE))
    & np.isfinite(raw_k0["k0_rel_ci_width"])
    & (raw_k0["k0_rel_ci_width"] <= K0_MAX_REL_CI_WIDTH)
    & np.isfinite(raw_k0["pH"])
    & np.isfinite(raw_k0["electrophile_pKa"])
    & (raw_k0["n_inliers"] >= K0_MIN_INLIERS)
)
if REQUIRE_STRATEGY_A_SUCCESS:
    raw_k0["k0_input_usable"] &= raw_k0["analysis_success"]

raw_k0["k0_input_exclusion_reason"] = np.select(
    [
        ~np.isfinite(raw_k0["k0_linear"]),
        raw_k0["k0_linear"] <= 0,
        ~raw_k0["k0_ci_finite"],
        ~raw_k0["k0_ci_ordered"],
        K0_REQUIRE_LO95_POSITIVE & ~raw_k0["k0_ci_positive"],
        ~np.isfinite(raw_k0["k0_rel_ci_width"]),
        raw_k0["k0_rel_ci_width"] > K0_MAX_REL_CI_WIDTH,
        ~np.isfinite(raw_k0["pH"]),
        ~np.isfinite(raw_k0["electrophile_pKa"]),
        raw_k0["n_inliers"] < K0_MIN_INLIERS,
        REQUIRE_STRATEGY_A_SUCCESS & ~raw_k0["analysis_success"],
    ],
    [
        "k0 missing/non-finite",
        "k0 <= 0",
        "k0 CI missing/non-finite",
        "k0 CI not ordered",
        "k0 lower 95% CI <= 0",
        "k0 relative CI width missing/non-finite",
        f"k0 relative CI width > {K0_MAX_REL_CI_WIDTH}",
        "pH missing/non-finite",
        "electrophile pKa missing/non-finite",
        f"n_inliers < {K0_MIN_INLIERS}",
        "strategy A k2 analysis did not pass",
    ],
    default="used",
)

read_errors = pd.DataFrame(errors)
usable_k0 = raw_k0[raw_k0["k0_input_usable"]].copy()

print(f"Read {len(raw_k0)} k0 rows")
print(f"Usable positive k0 rows = {len(usable_k0)}")
display(usable_k0.groupby("electrophile_type").size().rename("n_usable_k0").reset_index())
if len(read_errors):
    display(read_errors)

Found 30 Stage 2-4 result workbooks
Read 1872 k0 rows
Usable positive k0 rows = 1165


,electrophile_type,n_usable_k0
0,P5,361
1,P6,230
2,P7,172
3,P8,250
4,P9,152


## Fit k0 pH-Rate Profiles

In [4]:
summary_rows = []
assignment_rows = []
bootstrap_rows = []

for electrophile, group in usable_k0.groupby("electrophile_type", dropna=False):
    group = group.sort_values(["pH", "experiment", "nu_id"]).reset_index(drop=True)
    pKa_values = group["electrophile_pKa"].dropna().unique()
    pKa = float(pKa_values[0]) if len(pKa_values) else np.nan
    px_output_dir = resolve_px_output_dir(electrophile)
    px_output_dir.mkdir(parents=True, exist_ok=True)
    plot_path = px_output_dir / f"k0_ph_rate_{sanitize_filename(electrophile)}.png"
    point_workbook_path = px_output_dir / f"k0_ph_rate_{sanitize_filename(electrophile)}.xlsx"

    row = {
        "electrophile_type": electrophile,
        "PX_dir": px_output_dir.name,
        "PX_output_dir": str(px_output_dir),
        "electrophile_pKa": pKa,
        "n_k0_total_usable": len(group),
        "n_experiments": group["experiment"].nunique(),
        "n_nucleophiles": group["nu_id"].nunique() if "nu_id" in group.columns else np.nan,
        "pH_min": group["pH"].min(),
        "pH_max": group["pH"].max(),
        "plot_path": str(plot_path),
        "point_workbook_path": str(point_workbook_path),
    }

    if not np.isfinite(pKa) or len(group) < MIN_POINTS:
        row.update({"fit_success": False, "failure_reason": "missing pKa or fewer than 3 k0 values"})
        summary_rows.append(row)
        continue

    fit = fit_k0_ph_rate_two_step(
        group["pH"].to_numpy(),
        group["k0_linear"].to_numpy(),
        pKa,
        y_lo=group["k0_lo95"].to_numpy(),
        y_hi=group["k0_hi95"].to_numpy(),
    )
    if not fit.get("success", False):
        row.update({"fit_success": False, "failure_reason": fit.get("failure_reason", "fit failed")})
        summary_rows.append(row)
        continue

    boots = bayesian_bootstrap_k0_ph_rate(fit, pKa, n_boot=BOOTSTRAP_N)
    low_lo, low_hi = quantile_ci(boots["k0_low"]) if len(boots) else (np.nan, np.nan)
    high_lo, high_hi = quantile_ci(boots["k0_high"]) if len(boots) else (np.nan, np.nan)

    high_mask = group["pH"] >= pKa + 1.0
    high_vals = group.loc[high_mask, "k0_linear"].astype(float)
    high_summary = {
        "n": int(high_vals.count()),
        "mean": float(high_vals.mean()) if high_vals.count() else np.nan,
        "median": float(high_vals.median()) if high_vals.count() else np.nan,
        "sd": float(high_vals.std(ddof=1)) if high_vals.count() > 1 else np.nan,
    }

    plot_k0_profile(electrophile, pKa, fit, boots, high_summary, plot_path)

    row.update({
        "fit_success": True,
        "failure_reason": np.nan,
        "k0_low": fit["k0_low"],
        "k0_low_lo95": low_lo,
        "k0_low_hi95": low_hi,
        "k0_high": fit["k0_high"],
        "k0_high_lo95": high_lo,
        "k0_high_hi95": high_hi,
        "r2_log": fit["r2_log"],
        "rmse_log": fit["rmse_log"],
        "n_fit_total": fit["n_total"],
        "n_fit_inliers": fit["n_inliers"],
        "n_fit_outliers": fit["n_outliers"],
        "bootstrap_n_success": len(boots),
        "bootstrap_n_skipped": boots.attrs.get("skipped", 0) if boots is not None else np.nan,
        "high_pH_threshold": pKa + 1.0,
        "high_pH_k0_n": high_summary["n"],
        "high_pH_k0_mean": high_summary["mean"],
        "high_pH_k0_median": high_summary["median"],
        "high_pH_k0_sd": high_summary["sd"],
    })
    summary_rows.append(row)

    tmp = group.copy()
    tmp["fit_inlier"] = fit["inlier_mask"]
    tmp["fit_outlier"] = ~fit["inlier_mask"]
    tmp["k0_pred"] = fit["pred_all"]
    assignment_rows.append(tmp)

    if len(boots):
        b = boots.copy()
        b["electrophile_type"] = electrophile
        bootstrap_rows.append(b)

k0_summary = pd.DataFrame(summary_rows).sort_values("electrophile_type").reset_index(drop=True)
k0_values_with_fit = pd.concat(assignment_rows, ignore_index=True) if assignment_rows else pd.DataFrame()
k0_bootstrap = pd.concat(bootstrap_rows, ignore_index=True) if bootstrap_rows else pd.DataFrame()

display(k0_summary)

,electrophile_type,PX_dir,PX_output_dir,electrophile_pKa,n_k0_total_usable,n_experiments,n_nucleophiles,pH_min,pH_max,plot_path,...,n_fit_total,n_fit_inliers,n_fit_outliers,bootstrap_n_success,bootstrap_n_skipped,high_pH_threshold,high_pH_k0_n,high_pH_k0_mean,high_pH_k0_median,high_pH_k0_sd
0,P5,P5,/Users/robertmayer/Desktop/mayerlab_local/phos...,4.50,361,6,28,1.927307,13.001634,/Users/robertmayer/Desktop/mayerlab_local/phos...,...,361,345,16,2000,0,5.50,346,0.000024,0.000024,0.000004
1,P6,P6,/Users/robertmayer/Desktop/mayerlab_local/phos...,5.72,230,6,25,4.086879,12.929208,/Users/robertmayer/Desktop/mayerlab_local/phos...,...,230,210,20,2000,0,6.72,193,0.000003,0.000001,0.000008
2,P7,P7,/Users/robertmayer/Desktop/mayerlab_local/phos...,2.55,172,6,27,0.497049,12.998707,/Users/robertmayer/Desktop/mayerlab_local/phos...,...,172,168,4,2000,0,3.55,164,0.000003,0.000002,0.000007
3,P8,P8,/Users/robertmayer/Desktop/mayerlab_local/phos...,2.74,250,6,28,0.497049,12.998279,/Users/robertmayer/Desktop/mayerlab_local/phos...,...,250,250,0,2000,0,3.74,233,0.000008,0.000004,0.000011
4,P9,P9,/Users/robertmayer/Desktop/mayerlab_local/phos...,3.28,152,6,24,0.675928,13.001634,/Users/robertmayer/Desktop/mayerlab_local/phos...,...,152,152,0,2000,0,4.28,148,0.000017,0.000005,0.000030


## Save Results

In [5]:
metadata = pd.DataFrame([
    {"parameter": "KINETICS_DIR", "value": str(KINETICS_DIR)},
    {"parameter": "P_KA_WORKBOOK", "value": str(P_KA_WORKBOOK)},
    {"parameter": "BOOTSTRAP_N", "value": BOOTSTRAP_N},
    {"parameter": "RANDOM_SEED", "value": RANDOM_SEED},
    {"parameter": "MIN_POINTS", "value": MIN_POINTS},
    {"parameter": "OUTLIER_FACTOR", "value": OUTLIER_FACTOR},
    {"parameter": "K0_MIN_INLIERS", "value": K0_MIN_INLIERS},
    {"parameter": "REQUIRE_STRATEGY_A_SUCCESS", "value": REQUIRE_STRATEGY_A_SUCCESS},
    {"parameter": "K0_REQUIRE_LO95_POSITIVE", "value": K0_REQUIRE_LO95_POSITIVE},
    {"parameter": "K0_MAX_REL_CI_WIDTH", "value": K0_MAX_REL_CI_WIDTH},
    {"parameter": "n_source_workbooks", "value": len(workbooks)},
    {"parameter": "n_raw_k0_rows", "value": len(raw_k0)},
    {"parameter": "n_usable_k0_rows", "value": len(usable_k0)},
])

written_workbooks = []

for _, summary_row in k0_summary.iterrows():
    electrophile = summary_row["electrophile_type"]
    point_workbook = Path(summary_row["point_workbook_path"])
    point_metadata = pd.concat(
        [
            metadata,
            pd.DataFrame([
                {"parameter": "point_electrophile_type", "value": electrophile},
                {"parameter": "point_output_dir", "value": summary_row["PX_output_dir"]},
            ]),
        ],
        ignore_index=True,
    )
    point_values = k0_values_with_fit[k0_values_with_fit["electrophile_type"].eq(electrophile)].copy()
    point_raw = raw_k0[raw_k0["electrophile_type"].eq(electrophile)].copy()
    point_pka = pka_df[pka_df["electrophile_type"].eq(electrophile)].copy()
    point_errors = read_errors
    if len(read_errors) and "source_workbook" in read_errors.columns:
        point_errors = read_errors[
            read_errors["source_workbook"].astype(str).str.contains(str(summary_row["PX_output_dir"]), regex=False)
        ].copy()

    write_k0_excel_workbook(
        point_workbook,
        k0_summary[k0_summary["electrophile_type"].eq(electrophile)].copy(),
        point_values,
        point_raw,
        point_pka,
        point_metadata,
        point_errors,
    )
    written_workbooks.append(point_workbook)

print("Saved per-PX workbooks:")
for path in written_workbooks:
    print(f"- {path}")
print("Saved per-PX plots:")
for path in k0_summary["plot_path"]:
    print(f"- {path}")


Saved per-PX workbooks:
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/k0_ph_rate_P5.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P6/k0_ph_rate_P6.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P7/k0_ph_rate_P7.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P8/k0_ph_rate_P8.xlsx
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P9/k0_ph_rate_P9.xlsx
Saved per-PX plots:
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P5/k0_ph_rate_P5.png
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P6/k0_ph_rate_P6.png
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P7/k0_ph_rate_P7.png
- /Users/robertmayer/Desktop/mayerlab_local/phosphorylation_publ/section5_6_kinetics/P8/k0_ph_rate_P8.p